In [1]:
import requests
import time
import pandas as pd
from datetime import *

In [11]:
now = datetime.now()
max_attempts = 5
# 格式化日期
for _ in range(max_attempts):
    datestr = now.strftime("%Y%m%d")
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={datestr}&selectType=ALL&response=json&_=1687956428483"
    response = requests.get(url)
    data = response.json()
    
    if data.get("total", 1) == 0:
        print(f"No data for date {datestr}, trying the previous day.")
        now -= timedelta(days=1)
        continue
    else:
        df = pd.DataFrame(data["data"], columns=data["fields"])
        print(df.head(5))
        break  # 成功获取数据后跳出循环

No data for date 20230708, trying the previous day.
     證券代號           證券名稱 外陸資買進股數(不含外資自營商) 外陸資賣出股數(不含外資自營商) 外陸資買賣超股數(不含外資自營商)  \
0  00715L      期街口布蘭特正2         2,783,000          374,000         2,409,000   
1   00893      國泰智能電動車          9,752,150           89,000         9,663,150   
2    2618  長榮航                 39,045,143       25,964,000        13,081,143   
3  00642U    期元大S&P石油              10,000           51,000           -41,000   
4   00885   富邦越南                6,297,800          275,000         6,022,800   

  外資自營商買進股數 外資自營商賣出股數 外資自營商買賣超股數   投信買進股數     投信賣出股數     投信買賣超股數    自營商買賣超股數  \
0         0         0          0        0          0           0  42,192,000   
1         0         0          0        0          0           0   2,402,018   
2         0         0          0  136,000  5,274,000  -5,138,000   2,729,349   
3         0         0          0        0          0           0  10,346,993   
4         0         0          0        0          0           0   

In [10]:
max_attempts = 5

# 獲取當前日期
now = datetime.now()

for _ in range(max_attempts):
    # 格式化日期
    datestr = now.strftime("%Y%m%d")
    url = f"https://www.twse.com.tw/rwd/zh/fund/T86?date={datestr}&selectType=ALL&response=json&_=1687956428483"
    print('for', url)
    try:
        response = requests.get(url)
        response.raise_for_status()  # 如果請求失敗，這將引發一個異常

        # 將JSON回應轉換成Python字典
        data = response.json()

        # 如果數據列表為空，則繼續嘗試前一天的數據
        if data.get("total", 1) == 0:
            print(f"No data for date {date_string}, trying the previous day.")
            now -= timedelta(days=1)
            time.sleep(5)
            continue

        # 從字典中取出"data"鍵的值，該值是一個二維列表
        # data_list = data["data"]

        # 將二維列表轉換成pandas DataFrame
        df = pd.DataFrame(data["data"], columns=data["fields"])

        # 其他代碼...

        break  # 成功获取数据后跳出循环

    except requests.RequestException as e:
        print(f"Failed to download data for date {date_string}: {e}")
    except Exception as e:
        print(f"Failed to process data for date {date_string}: {e}")

    # 減少一天並重新嘗試
    now -= timedelta(days=1)
    time.sleep(5)  # 避免过于频繁的请求

else:  # 这个 else 与 for 对应，当 for 循环正常结束（没有被 break 中断）时执行
    print("Failed to download data after multiple attempts.")


for https://www.twse.com.tw/rwd/zh/fund/T86?date=20230708&selectType=ALL&response=json&_=1687956428483
No data for date 20230708, trying the previous day.
Failed to process data for date 20230708: type object 'datetime.time' has no attribute 'sleep'


AttributeError: type object 'datetime.time' has no attribute 'sleep'

In [ ]:
# 将JSON回應转换为Python字典
data = response.json()
data_list = data["data"]
df = pd.DataFrame(data_list, columns=data["fields"])
df_for = df[['證券代號','證券名稱','外陸資買賣超股數(不含外資自營商)']].copy()
df_for['外陸資買賣超股數(不含外資自營商)'] = df_for['外陸資買賣超股數(不含外資自營商)'].str.replace(',', '')
df_for['外陸資買賣超股數(不含外資自營商)'] = pd.to_numeric(df_for['外陸資買賣超股數(不含外資自營商)'], errors='coerce')
df_for_new = df_for[df_for['外陸資買賣超股數(不含外資自營商)'].notna() & (df_for['外陸資買賣超股數(不含外資自營商)'] != 0)]
df_for_new = df_for_new.sort_values('外陸資買賣超股數(不含外資自營商)', ascending=False)
df_for_new2 = df_for_new.sort_values('外陸資買賣超股數(不含外資自營商)', ascending=True)
df_buy_top50 = df_for_new.head(50)
df_sell_top50 = df_for_new2.head(50)